## TODOs: Chunking (Step 4)

- Set up imports: `HybridChunker`, `HuggingFaceTokenizer`, `AutoTokenizer`, `DoclingDocument`
- Configure `HybridChunker`: `tokenizer`, `max_tokens=512`, `merge_peers=True`
- Load the JSON saved in Step 3 back into a `DoclingDocument` object via `DoclingDocument.load_from_json(path)` (no re-parsing of the PDF)
- Run `chunker.chunk(dl_doc=doc)`, iterate over chunks, apply `chunker.contextualize()`
- Inspect chunks: count, token distribution (min/max/median), spot-check samples, check shortest/degenerate chunks
- Verify that bbox/page_no metadata is preserved per chunk (`chunk.meta.doc_items`)
- Define a Pydantic schema for a chunk in `schemas.py` (`chunk_id`, `doc_id`, `text`, `contextualized_text`, `page_no`, `bbox`, `heading_path`, `token_count`)
- Generate stable, reproducible chunk IDs
- Save chunks as JSONL (basis for caching in Step 5 and embedding in Step 6)
- Once everything works in the notebook → move the logic into `ingestion.py`

In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import json
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types.doc import DoclingDocument
from pathlib import Path
from transformers import AutoTokenizer
from backend.config import settings
import jsonlines
import backend.doc_processing as hf

In [2]:
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained("BAAI/bge-m3"),
    max_tokens=512,  # explizit setzen, sonst wird model_max_length verwendet
)
hyb_chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

In [3]:
print(settings.DOCUMENTS_OUT_DIR)

C:\Users\admin\Programming\RAG-Systems\classic-RAG\Classic-RAG\data\output_docs


In [ ]:
hashes = hf.hash_files(settings.DOCUMENTS_IN_DIR)

settings.CHUNKS_OUT_DIR.mkdir(parents=True, exist_ok=True)
with jsonlines.open(settings.CHUNKS_OUT_DIR/"chunks.jsonl", mode="w") as writer:
    
    for stem, doc_id in hashes:
        fp = list(settings.DOCUMENTS_OUT_DIR.glob(f"{stem}.json"))[0]
        doc = DoclingDocument.load_from_json(fp)
        
        for i,chunk in enumerate(hyb_chunker.chunk(dl_doc=doc)):
            total = chunk.model_dump(mode="json")
            total["chunk_id"] = f"{i}"
            total["contextualized_text"] = hyb_chunker.contextualize(chunk)
            total["doc_id"] = f"{doc_id}"
            writer.write(total)
    